#Ingestion

In [1]:
from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content="Hey is this working?",
    metadata = {
        "source":"try.txt",
        "author":"Nazam kalsi",
        "page":1,
        "created_at":"25-03-2026"
    }
)
doc

Document(metadata={'source': 'try.txt', 'author': 'Nazam kalsi', 'page': 1, 'created_at': '25-03-2026'}, page_content='Hey is this working?')

In [4]:
import os
os.makedirs("../data/txtFiles",exist_ok=True)
text={
    "../data/txtFiles/text01.txt":"""Evolving Engineer
Evolving Engineer


AI 🤖
Everything you need to know about production ready RAG systems : Part 1
Deep-diving into document ingestion, chunking strategies, and the architecture of real-world retrieval systems
Pradyumna
Nov 18, 2025

Everything you need to know about Production ready RAG systems: Part 2
Everything you need to know about Production ready RAG systems: Part 2
Pradyumna
·
November 23, 2025
Read full story
Everything you need to know about production ready RAG systems : Part 3
Everything you need to know about production ready RAG systems : Part 3
Pradyumna
·
November 28, 2025
Read full story
Appreciate you reading this newsletter, I hope the content and resources are helping you level up as an engineer. If you want me to write about something specific, just message me or comment on the post.

Note: This article on was originally published on my blog - https://pradyumnachippigiri.dev/blogs/rag-systems-part1

What is RAG?
RAG stands for Retrieval Augmented Generation. It was introduced in the paper Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.

Each step can be roughly broken down to:

Retrieval – Seeking relevant information from a source given a query. For example, getting relevant passages of Wikipedia text from a database given a question.

Augmented – Using the relevant retrieved information to modify an input to a generative model (e.g., an LLM).

Generation – Generating an output given an input. For example, in the case of an LLM, generating a passage of text given an input prompt.

Thanks for reading LevelYouUp! Subscribe for free to receive new posts and support my work.

Type your email...
Subscribe
You can think of RAG like an open-book exam:

You or your mind is the LLM.

The textbook is your external knowledge base (PDFs, docs, webpages).

When a question is asked, you don’t memorize the entire book, instead, you first flip through the index/table of contents to find the most relevant pages → This “flipping and finding the right pages” is Retrieval.

Once you have the right pages open, you read them and augment your understanding with that information → This is Augmentation.

Finally, based on the question + the retrieved pages, you think, synthesize, and produce a well-structured answer → This is Generation.

So the pipeline is:

Index search → retrieve relevant pages → read them → think → answer.

Exactly how RAG works:

Retriever: finds relevant chunks like an exam index

Augmentor: inserts those chunks into the model’s prompt

LLM Generator: reasons using the question + retrieved context

retrieval_diagram


You might now ask, our mind is not capable to read the entire book to answer quickly to the question, but LLMs can right, so what is the use of RAG ?
I asked ChatGPT and Google Gemini to answer questions from a 1000 page pdf and here’s how it answered.




chatgpt answer



gemini answer

As you can see that Chatgpt was not able to answer as its context window is lesser compared to Gemini. Hence, Google Gemini was able to read and output the question asked.
But, here comes the catch: Even though LLMs today have massive context windows and can technically “read” huge amounts of text, there are three major limitations that make RAG still essential.

Even though LLMs today have massive context windows and can technically “read” huge amounts of text, there are two major limitations that make RAG still essential.

- Cost explodes with large context windows:
LLMs charge per input + output token.
Sending a 200K-token PDF into the model every time is insanely expensive.

- Attention quality drops as context increases:
LLMs lose accuracy and focus when you overload them.
RAG narrows the context → better reasoning.

So RAG is valuable and will be in the future too

Thanks for reading LevelYouUp! Subscribe for free to receive new posts and support my work.

Type your email...
Subscribe
The Four Main Building Blocks of a RAG System
RAG Pipeline


RAG pipeline
In this article we will talk about Document Preprocessing that is basically Data Ingestion, and Data Chunking and different strategies in chunking. In the next article we will talk about Embedding generation and retrieval.

Data Ingestion
Everyone thinks Data Ingestion is easy “just upload a PDF and extract text” but there are so many complexities involved in this as well, If given a PDF, where would you store it, how would you parse it? How would you deal with images and tables?

There are some powerful libraries that you should know:

PyMuPDF (fitz)
Best for mixed PDFs (text + images).
Extracts:

text

images

metadata

layout

Docs: https://pymupdf.readthedocs.io/en/latest/

But if a page is JUST a scanned image, PyMuPDF cannot read text → you need OCR.

OCR using Tesseract
For scanned PDFs or images:

Reads text from images

Works for scanned PDFs

Not ideal for complex tables or multi-column layouts

Repo: https://github.com/tesseract-ocr/tesseract

Docling
Extracts:

rows

columns

table schemas

layout

and converts everything into clean JSON

Docs: https://docling-project.github.io/docling/

These libraries help you in opening and reading the pdfs, but in some cases when you are dealing with industry level projects, often you do not have the pdfs, you will have to scrape the data from the web. So here are some libraries you would want to know to scrape data from the web.

Firecrawl
Fully crawls:

websites

blogs

documentation

knowledge bases

…and converts them into LLM-ready Markdown.

Docs - https://www.firecrawl.dev/

Puppeteer
Think of it as an automated browser that you can script the same way you would manually click, search, scroll, and interact with web pages.
Also lets you scrape only what you want:

<p> tags

headers

titles

selected divs

links to all PDFs

and even download 500 PDFs automatically

Docs - https://pptr.dev/

Data Chunking
Once your documents are cleaned and extracted, the next question is: “How should I split the text so the LLM can retrieve the right information?”

Chunking matters because:

too large = irrelevant info

too small = lost context

badly aligned = wrong retrieval

There are five chunking strategies, each with specific use cases and each of them have their own pros and cons, lets discuss all of that in this discussion :

Fixed-Size Chunking
Fixed-size chunking divides text into chunks based purely on lengt, for example, every 200 words or every 500 tokens. It simply chops the text based on size rules, which makes it extremely fast but contextually weak.

Pros:

Very fast and easy to implement

Works well for large volumes of short, noisy text

Cons:

Breaks semantic flow (chunks may start mid-sentence)

Context can get lost across boundaries

Best For:

Social media data (reddit, twitter etc.)

Logs

Short, unstructured content where coherence is less important

Semantic Chunking

Semantic Chunking


semantic chunking
Semantic chunking uses embedding similarity to group sentences that are meaningfully related. Instead of breaking text by size, it breaks based on content. Each sentence is compared to the chunk being built; if it stays semantically similar, it joins the chunk. If it drifts too far than the cosine similarity threshold that we mention then, a new chunk is created.

Pros:

Produces highly coherent, context-rich chunks

Great for deep retrieval accuracy

Cons:

Slow (needs embeddings for every sentence)

Chunk sizes can vary (some can be tiny, others huge)

Best For:

Legal contracts

Policies, compliance docs

Structural Chunking
Structural chunking uses the natural format of the document headings, subheadings, sections, tables of contents to define chunk boundaries. It respects the author’s intended organization, making chunks more meaningful and human-readable. Example by Introduction, Overview, Results etc.

Pros:

Very clean, interpretable chunks

Fast and consistent

Preserves metadata like titles and sections

Cons:

Sections may vary widely in length hence chunk sizes can vary immensely.

Not ideal for documents with poor or missing structure

Best For:

PDFs with headings

Research papers

Reports, guides, manuals

Technical Documentations

Sliding-window chunking
Sliding-window chunking creates overlapping chunks to preserve continuity. Instead of splitting text once, you slide a window across the document for example, a 500-token window moving forward 250 tokens at a time. This ensures that important context that appears near chunk boundaries is not lost. Example:
Chunk 1: tokens 0–500
Chunk 2: tokens 250–750
Chunk 3: tokens 500–1000

Pros:

Maintains continuity across chunk boundaries

Reduces “edge-case” information loss

Improves retrieval for sequential content

Cons:

Higher storage requirements

More redundant data to embed and store

Best For:

Transcripts

Podcasts

Dialogue/conversations

Any long, sequential content

Recursive chunking

Recursive Chunking


Recursive Chunking
Recursive chunking is a hybrid strategy. It first splits the document using high-level structure (e.g., headings). Then, if any section exceeds a maximum size that we mention, it splits that section again using another method (often fixed-size or semantic). This maintains both structure and consistency.

Pros:

Produces consistently sized chunks

Still respects hierarchy and structure

Ideal for large, messy documents

Cons:

More complex to implement

Requires choosing multiple chunking rules

Best For:

Enterprise-grade RAG

Long, heterogeneous PD

Thanks for reading LevelYouUp! Subscribe for free to receive new posts and support my work.

Type your email...
Subscribe
Which chunking strategy to use and when ?
The image belwo shows exacly when we should prefer one over the pther.




the most important question to ask is: “Does my data need chunking at all?”

Chunking is designed to break down long, unstructured documents. If your data source already has small, complete pieces of information like FAQs, product descriptions, or social media posts, you usually do not need to chunk them. Chunking can even cause problems. The goal is to create meaningful semantic units, and if your data is already in that format, you’re ready for the embedding stage.

Once you’ve confirmed that your documents are long enough to benefit from chunking, you can use the following questions to guide your choice of strategy:

What is the nature of my documents? Are they highly structured (like code or JSON), or are they unstructured narrative text?

What level of detail does my RAG system need? Does it need to retrieve specific, granular facts or summarize broader concepts?

Which embedding model am I using? What are the size of the output vectors (more dimensions increases the ability for more granular information to be stored)??

How complex will my user queries be? Will they be simple questions that need small, targeted chunks, or complex ones that require more context?

How to Optimize The Chunk Size for RAG in Production
Optimizing chunk size in a production setting takes many tests and reviews. Here are some steps you can take:

Begin with a common baseline strategy, such as fixed-size chunking. A good place to start is a chunk size of 512 tokens and a chunk overlap of 50-100 tokens. This gives you a solid baseline that’s easy to reproduce and compare other chunking strategies against.

Experiment with different chunking approaches by tweaking parameters like chunk size and overlap to find what works best for your data.

Test how well your retrieval works by running typical queries and checking metrics like hit rate, precision, and recall to see which strategy delivers.

Involve humans to review both the retrieved chunks and LLM-generated responses - their feedback will catch things metrics might miss.

Continuously monitor the performance of your RAG system in production and be prepared to iterate on your chunking strategy as needed.

Already this article has gone sorry for that, in the next article we will learn about Embeddings and Generation part of it.
Hope you liked this article, and if you liked it please do leave a like.
See you in the next one.

Thanks for reading LevelYouUp! Subscribe for free to receive new posts and support my work.

Type your email...
Subscribe
31 Likes
∙
11 Restacks
Discussion about this post
Write a comment...
Harsh Chandwani
Nov 22

One of the best article around this.

Like (1)
Reply
Share
1 reply by Pradyumna
1 more comment...

Apache Kafka Basics
Understanding the core components of Kafka for System design interviews
Feb 9 • Pradyumna

53

4

9


Client-Side Proxy vs Server-Side Proxy
Understanding forward proxies and reverse proxies from first principles, plus how they differ from VPNs and firewalls
Jan 25 • Pradyumna

9

1

3


How databases store data on the disk
Easy explanation of how HDDs and SSDs store data from database
Feb 19 • Pradyumna

8


2


Ready for more?
Type your email...
Subscribe
© 2026 Pradyumna · Privacy ∙ Terms ∙ Collection notice
Start your Substack
Get the app
Substack is the home for great culture
"""    
}

for f,c in text.items():
    with open(f,"w",encoding="utf-8") as file:
        file.write(c)

    

In [6]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader("../data/doc.docx");
data = loader.load()
data

[Document(metadata={'source': '../data/doc.docx'}, page_content='Buttons\n\nButtons\n\nStatus\n\nStatus\n\nVoltage Status\n\nVoltage Status\n\nVoltage Status\n\nVoltage Status\n\nButtons\n\nButtons\n\n\n\n`\n\n\n\nCommands:-\n\nSEISMIC ACTIVATE\n\nSEISMIC DEACTIVATE\n\nMANUAL ARM \n\nMANUAL DISARM \n\nSELF NEUTRALIZATION ENGAGE\n\nFIRE\n\nSAFE\n\nHEALTH\n\n\n\nSTATUSES:-\n\nSEISMIC ACTIVE\n\nSEISMIC ARMED\n\nMANIAL ARM ACTIVE\n\nTILT ANGLE\n\n\n\nTECHNICAL\n\nTECHNOLOGIES USED:-\n\nPYTHON\n\nPYQT\n\nFOLIUM MAPS\n\nCODE EXPLANATION:-\n\nBUTTONS INITIALIZATION:-\n\n\n\n\n\n\n\nCOMMANDS HANDLING:-\n\n\n\n\n\nSerial Port Initialization:-')]

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader

loader = UnstructuredExcelLoader("../excel.xlsx")
docs = loader.load()


ModuleNotFoundError: No module named 'msoffcrypto'